In [ ]:
import pandas as pd
import numpy as np
import nltk
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score,precision_score, recall_score, f1_score,classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
import torch
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
data = pd.read_csv("/content/processed_data.csv")
data.head()

,QuestionText,Category,Answer,Question_for_classification,Answer_for_classification,Question_for_QA,Answer_for_QA,Question_for_translation,Answer_for_translation,Question_tokens,Answer_tokens,Question_tokens_no_stopwords,Answer_tokens_no_stopwords,Question_tokens_stemmed,Answer_tokens_stemmed,Question_text_no_stopwords,Question_text_stemmed,Answer_text_no_stopwords,Answer_text_stemmed
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,ايهما افضل الدراسة في السابق ام في الوقت الحالي,الدراسة في الوقت الحالي تعتبر افضل بسبب توفر ا...,ايهما افضل الدراسة في السابق ام في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر افضل بسبب توفر ا...,ايهما افضل الدراسة في السابق ام في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر افضل بسبب توفر ا...,"['ايهما', 'افضل', 'الدراسة', 'في', 'السابق', '...","['الدراسة', 'في', 'الوقت', 'الحالي', 'تعتبر', ...","['ايهما', 'افضل', 'الدراسة', 'السابق', 'ام', '...","['الدراسة', 'الوقت', 'الحالي', 'تعتبر', 'افضل'...","['ايه', 'فضل', 'درس', 'سبق', 'ام', 'وقت', 'الح...","['درس', 'وقت', 'الحالي', 'عبر', 'فضل', 'سبب', ...",ايهما افضل الدراسة السابق ام الوقت الحالي,ايه فضل درس سبق ام وقت الحالي,الدراسة الوقت الحالي تعتبر افضل بسبب توفر التك...,درس وقت الحالي عبر فضل سبب وفر كنولوج ورد علم حدث
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,اليس القطن عماد الثروة في مصر,القطن يعتبر من اهم المنتجات الزراعية في مصر وي...,اليس القطن عماد الثروة في مصر؟,القطن يعتبر من اهم المنتجات الزراعية في مصر، و...,اليس القطن عماد الثروة في مصر؟,القطن يعتبر من اهم المنتجات الزراعية في مصر، و...,"['اليس', 'القطن', 'عماد', 'الثروة', 'في', 'مصر']","['القطن', 'يعتبر', 'من', 'اهم', 'المنتجات', 'ا...","['اليس', 'القطن', 'عماد', 'الثروة', 'مصر']","['القطن', 'يعتبر', 'من', 'اهم', 'المنتجات', 'ا...","['الس', 'قطن', 'عمد', 'ثرة', 'مصر']","['قطن', 'عبر', 'من', 'اهم', 'نتج', 'زرع', 'مصر...",اليس القطن عماد الثروة مصر,الس قطن عمد ثرة مصر,القطن يعتبر من اهم المنتجات الزراعية مصر ويعد ...,قطن عبر من اهم نتج زرع مصر يعد من عمد ريس قصد صري
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.,اتصعد الشمس من الشرق,الشمس تصعد من الشرق,اتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.,اتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.,"['اتصعد', 'الشمس', 'من', 'الشرق']","['الشمس', 'تصعد', 'من', 'الشرق']","['اتصعد', 'الشمس', 'من', 'الشرق']","['الشمس', 'تصعد', 'من', 'الشرق']","['صعد', 'شمس', 'من', 'شرق']","['شمس', 'صعد', 'من', 'شرق']",اتصعد الشمس من الشرق,صعد شمس من شرق,الشمس تصعد من الشرق,شمس صعد من شرق
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.,اتعرف البكتيريا بانها كاينات حية دقيقة,البكتيريا تعرف بانها كاينات حية دقيقة,اتعرف البكتيريا بانها كاينات حية دقيقة؟,البكتيريا تعرف بانها كاينات حية دقيقة.,اتعرف البكتيريا بانها كاينات حية دقيقة؟,البكتيريا تعرف بانها كاينات حية دقيقة.,"['اتعرف', 'البكتيريا', 'بانها', 'كاينات', 'حية...","['البكتيريا', 'تعرف', 'بانها', 'كاينات', 'حية'...","['اتعرف', 'البكتيريا', 'بانها', 'كاينات', 'حية...","['البكتيريا', 'تعرف', 'بانها', 'كاينات', 'حية'...","['عرف', 'كتر', 'بان', 'كين', 'حية', 'دقق']","['كتر', 'عرف', 'بان', 'كين', 'حية', 'دقق']",اتعرف البكتيريا بانها كاينات حية دقيقة,عرف كتر بان كين حية دقق,البكتيريا تعرف بانها كاينات حية دقيقة,كتر عرف بان كين حية دقق
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.,ايتكون الهوا اساسا من النيتروجين,الهوا يتكون اساسا من النيتروجين,ايتكون الهوا اساسا من النيتروجين؟,الهوا يتكون اساسا من النيتروجين.,ايتكون الهوا اساسا من النيتروجين؟,الهوا يتكون اساسا من النيتروجين.,"['ايتكون', 'الهوا', 'اساسا', 'من', 'النيتروجين']","['الهوا', 'يتكون', 'اساسا', 'من', 'النيتروجين']","['ايتكون', 'الهوا', 'اساسا', 'من', 'النيتروجين']","['الهوا', 'يتكون', 'اساسا', 'من', 'النيتروجين']","['ايت', 'هوا', 'سسا', 'من', 'ترج']","['هوا', 'يتك', 'سسا', 'من', 'ترج']",ايتكون الهوا اساسا من النيتروجين,ايت هوا سسا من ترج,الهوا يتكون اساسا من النيتروجين,هوا يتك سسا من ترج


In [ ]:
data['Category'].value_counts()

,count
Category,
الثقافة,725
التعليم,600
العلوم,597
الصحة,537
البيئة والطاقة,500
التكنولوجيا,439
التاريخ,402
الاقتصاد والعمل,290
الجغرافيا,203


In [ ]:
le = LabelEncoder()
y = le.fit_transform(data["Category"])

In [ ]:
X_bert = np.load("/content/bert_embeddings.npy")
X_qwen = np.load("/content/qwen_embeddings.npy")
X_e5 = np.load("/content/e5_embeddings.npy")
X_bge = np.load("/content/bge_embeddings.npy")

X_fasttext_no_stop = np.load("/content/fasttext_no_stop_embeddings.npy")
X_fasttext_stemmed = np.load("/content/fasttext_stemmed_embeddings.npy")

X_sg_no_stop = np.load("/content/sg_no_stop_embeddings.npy")
X_sg_stemmed = np.load("/content/sg_stemmed_embeddings.npy")

X_cbow_no_stop = np.load("/content/cbow_no_stop_embeddings.npy")
X_cbow_stemmed = np.load("/content/cbow_stemmed_embeddings.npy")

X_sg_trained = np.load("/content/sg_trained_embeddings.npy")
X_cbow_trained = np.load("/content/cbow_trained_embeddings.npy")

In [ ]:
embedding_features = {
    "BERT": X_bert,
    "QWEN": X_qwen,
    "E5": X_e5,
    "BGE": X_bge,
    "FastText No Stop": X_fasttext_no_stop,
    "FastText Stemmed": X_fasttext_stemmed,
    "SG No Stop": X_sg_no_stop,
    "SG Stemmed": X_sg_stemmed,
    "CBOW No Stop": X_cbow_no_stop,
    "CBOW Stemmed": X_cbow_stemmed,
    "Word2Vec SG Trained": X_sg_trained,
    "Word2Vec CBOW Trained": X_cbow_trained
}

In [ ]:
models = {
    "SVM": SVC(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "GaussianNB": GaussianNB(),
    "Random Forest": RandomForestClassifier(random_state=42)
}

In [ ]:
results = []
for feature_name, X in embedding_features.items():
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        results.append({
            "Feature": feature_name,
            "Model": model_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, average="weighted"),
            "Recall": recall_score(y_test, y_pred, average="weighted"),
            "F1-score": f1_score(y_test, y_pred, average="weighted")
        })

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Accuracy",ascending=False).reset_index(drop=True)
results_df

,Feature,Model,Accuracy,Precision,Recall,F1-score
0,QWEN,SVM,0.784456,0.788256,0.784456,0.782389
1,E5,Logistic Regression,0.757513,0.758434,0.757513,0.756245
2,BERT,SVM,0.756477,0.760221,0.756477,0.751870
3,BERT,Logistic Regression,0.754404,0.754319,0.754404,0.753040
4,BGE,SVM,0.752332,0.754335,0.752332,0.747765
5,BGE,Logistic Regression,0.736788,0.736757,0.736788,0.735551
6,QWEN,Logistic Regression,0.720207,0.721877,0.720207,0.719426
7,E5,SVM,0.713990,0.712397,0.713990,0.702381
8,BGE,GaussianNB,0.709845,0.725882,0.709845,0.708495
9,QWEN,GaussianNB,0.676684,0.687375,0.676684,0.675571


In [ ]:
results_df.to_csv("classification_embeddings_results.csv", index=False)

TF-IDF

In [ ]:
# TF-IDF on cleaned text
vectorizer = TfidfVectorizer()
tfidf_clean = vectorizer.fit_transform(data['Question_for_classification'])
print(tfidf_clean.shape)

(4821, 5952)


In [ ]:
# TF-IDF on no stopwords text
vectorizer = TfidfVectorizer()
tfidf_no_stop = vectorizer.fit_transform(data['Question_text_no_stopwords'])
print(tfidf_no_stop.shape)

(4821, 5856)


In [ ]:
# TF-IDF on stemmed text
vectorizer = TfidfVectorizer()
tfidf_stemmed = vectorizer.fit_transform(data['Question_text_stemmed'])
print(tfidf_stemmed.shape)

(4821, 2414)


BOW

In [ ]:
# BoW Unigram on cleaned text
cv = CountVectorizer(max_features=500,ngram_range=(1,1))
cv1_clean = cv.fit_transform(data['Question_for_classification'])
print(cv1_clean.shape)

(4821, 500)


In [ ]:
# BoW Unigram on no stopwords text
cv = CountVectorizer(max_features=500,ngram_range=(1,1))
cv1_no_stop = cv.fit_transform(data['Question_text_no_stopwords'])
print(cv1_no_stop.shape)

(4821, 500)


In [ ]:
# BoW Unigram on stemmed text
cv = CountVectorizer(max_features=500,ngram_range=(1,1))
cv1_stemmed = cv.fit_transform(data['Question_text_stemmed'])
print(cv1_stemmed.shape)

(4821, 500)


In [ ]:
#  Bigram on cleaned text
cv = CountVectorizer(max_features=500,ngram_range=(2,2))
cv2_clean = cv.fit_transform(data['Question_for_classification'])
print(cv2_clean.shape)

(4821, 500)


In [ ]:
#  Bigram on no stopwords text
cv = CountVectorizer(max_features=500,ngram_range=(2,2))
cv2_no_stop = cv.fit_transform(data['Question_text_no_stopwords'])
print(cv2_no_stop.shape)

(4821, 500)


In [ ]:
#  Bigram on stemmed text
cv = CountVectorizer(max_features=500,ngram_range=(2,2))
cv2_stemmed = cv.fit_transform(data['Question_text_stemmed'])
print(cv2_stemmed.shape)

(4821, 500)


In [ ]:
# Trigram on cleaned text
cv = CountVectorizer(max_features=500,ngram_range=(3,3))
cv3_clean = cv.fit_transform(data['Question_for_classification'])
print(cv3_clean.shape)

(4821, 500)


In [ ]:
# Trigram on no stopwords text
cv = CountVectorizer(max_features=500,ngram_range=(3,3))
cv3_no_stop = cv.fit_transform(data['Question_text_no_stopwords'])
print(cv3_no_stop.shape)

(4821, 500)


In [ ]:
# Trigram on stemmed text
cv = CountVectorizer(max_features=500,ngram_range=(3,3))
cv3_stemmed = cv.fit_transform(data['Question_text_stemmed'])
print(cv3_stemmed.shape)

(4821, 500)


In [ ]:
traditional_features = {
    "TF-IDF Clean": tfidf_clean,
    "TF-IDF No Stop": tfidf_no_stop,
    "TF-IDF Stemmed": tfidf_stemmed,

    "BoW Clean Unigram": cv1_clean,
    "BoW No Stop Unigram": cv1_no_stop,
    "BoW Stemmed Unigram": cv1_stemmed,

    "BoW Clean Bigram": cv2_clean,
    "BoW No Stop Bigram": cv2_no_stop,
    "BoW Stemmed Bigram": cv2_stemmed,

    "BoW Clean Trigram": cv3_clean,
    "BoW No Stop Trigram": cv3_no_stop,
    "BoW Stemmed Trigram": cv3_stemmed
}

In [ ]:
results = []

for feature_name, X in traditional_features.items():
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

    for model_name, model in models.items():

        if model_name == "GaussianNB":
            X_train_input = X_train.toarray()
            X_test_input = X_test.toarray()
        else:
            X_train_input = X_train
            X_test_input = X_test

        model.fit(X_train_input, y_train)
        y_pred = model.predict(X_test_input)

        results.append({
            "Technique": feature_name,
            "Classifier": model_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, average="weighted"),
            "Recall": recall_score(y_test, y_pred, average="weighted"),
            "F1-score": f1_score(y_test, y_pred, average="weighted")
        })

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [ ]:
traditional_results_df = pd.DataFrame(results)
traditional_results_df = traditional_results_df.sort_values(by="Accuracy",ascending=False).reset_index(drop=True)
traditional_results_df

,Technique,Classifier,Accuracy,Precision,Recall,F1-score
0,TF-IDF Stemmed,SVM,0.699482,0.718652,0.699482,0.697619
1,TF-IDF Stemmed,Logistic Regression,0.683938,0.697894,0.683938,0.678477
2,BoW Stemmed Unigram,Logistic Regression,0.667358,0.670302,0.667358,0.663745
3,TF-IDF Stemmed,Random Forest,0.659067,0.671626,0.659067,0.657541
4,TF-IDF Clean,Logistic Regression,0.655959,0.679162,0.655959,0.648973
5,TF-IDF No Stop,Logistic Regression,0.654922,0.678104,0.654922,0.647503
6,BoW Stemmed Unigram,SVM,0.650777,0.664829,0.650777,0.646702
7,TF-IDF No Stop,SVM,0.650777,0.683611,0.650777,0.646813
8,BoW Stemmed Unigram,Random Forest,0.644560,0.649141,0.644560,0.642689
9,TF-IDF Clean,SVM,0.640415,0.675182,0.640415,0.636322


The best classifier was SVM.
The best text representation was TF-IDF with stemming.
The best overall combination was TF-IDF Stemmed + SVM, achieving 68.9% accuracy.
Stemming improved most experiments.
Locally trained Word2Vec embeddings performed poorly due to limited dataset size.
 Bag of Words: unigram achieved the best performance, followed by bigram, while trigram produced the lowest results due to feature sparsity and limited dataset size.


In [ ]:
!pip install transformers accelerate

In [ ]:
from datasets import Dataset
from transformers import (AutoTokenizer,AutoModelForSequenceClassification,Trainer,TrainingArguments)

In [ ]:
data = pd.read_csv("processed_data.csv")
data.head()

,QuestionText,Category,Answer,Question_for_classification,Answer_for_classification,Question_for_QA,Answer_for_QA,Question_for_translation,Answer_for_translation,Question_tokens,Answer_tokens,Question_tokens_no_stopwords,Answer_tokens_no_stopwords,Question_tokens_stemmed,Answer_tokens_stemmed,Question_text_no_stopwords,Question_text_stemmed,Answer_text_no_stopwords,Answer_text_stemmed
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,ايهما افضل الدراسة في السابق ام في الوقت الحالي,الدراسة في الوقت الحالي تعتبر افضل بسبب توفر ا...,ايهما افضل الدراسة في السابق ام في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر افضل بسبب توفر ا...,ايهما افضل الدراسة في السابق ام في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر افضل بسبب توفر ا...,"['ايهما', 'افضل', 'الدراسة', 'في', 'السابق', '...","['الدراسة', 'في', 'الوقت', 'الحالي', 'تعتبر', ...","['ايهما', 'افضل', 'الدراسة', 'السابق', 'ام', '...","['الدراسة', 'الوقت', 'الحالي', 'تعتبر', 'افضل'...","['ايه', 'فضل', 'درس', 'سبق', 'ام', 'وقت', 'الح...","['درس', 'وقت', 'الحالي', 'عبر', 'فضل', 'سبب', ...",ايهما افضل الدراسة السابق ام الوقت الحالي,ايه فضل درس سبق ام وقت الحالي,الدراسة الوقت الحالي تعتبر افضل بسبب توفر التك...,درس وقت الحالي عبر فضل سبب وفر كنولوج ورد علم حدث
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,اليس القطن عماد الثروة في مصر,القطن يعتبر من اهم المنتجات الزراعية في مصر وي...,اليس القطن عماد الثروة في مصر؟,القطن يعتبر من اهم المنتجات الزراعية في مصر، و...,اليس القطن عماد الثروة في مصر؟,القطن يعتبر من اهم المنتجات الزراعية في مصر، و...,"['اليس', 'القطن', 'عماد', 'الثروة', 'في', 'مصر']","['القطن', 'يعتبر', 'من', 'اهم', 'المنتجات', 'ا...","['اليس', 'القطن', 'عماد', 'الثروة', 'مصر']","['القطن', 'يعتبر', 'من', 'اهم', 'المنتجات', 'ا...","['الس', 'قطن', 'عمد', 'ثرة', 'مصر']","['قطن', 'عبر', 'من', 'اهم', 'نتج', 'زرع', 'مصر...",اليس القطن عماد الثروة مصر,الس قطن عمد ثرة مصر,القطن يعتبر من اهم المنتجات الزراعية مصر ويعد ...,قطن عبر من اهم نتج زرع مصر يعد من عمد ريس قصد صري
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.,اتصعد الشمس من الشرق,الشمس تصعد من الشرق,اتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.,اتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.,"['اتصعد', 'الشمس', 'من', 'الشرق']","['الشمس', 'تصعد', 'من', 'الشرق']","['اتصعد', 'الشمس', 'من', 'الشرق']","['الشمس', 'تصعد', 'من', 'الشرق']","['صعد', 'شمس', 'من', 'شرق']","['شمس', 'صعد', 'من', 'شرق']",اتصعد الشمس من الشرق,صعد شمس من شرق,الشمس تصعد من الشرق,شمس صعد من شرق
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.,اتعرف البكتيريا بانها كاينات حية دقيقة,البكتيريا تعرف بانها كاينات حية دقيقة,اتعرف البكتيريا بانها كاينات حية دقيقة؟,البكتيريا تعرف بانها كاينات حية دقيقة.,اتعرف البكتيريا بانها كاينات حية دقيقة؟,البكتيريا تعرف بانها كاينات حية دقيقة.,"['اتعرف', 'البكتيريا', 'بانها', 'كاينات', 'حية...","['البكتيريا', 'تعرف', 'بانها', 'كاينات', 'حية'...","['اتعرف', 'البكتيريا', 'بانها', 'كاينات', 'حية...","['البكتيريا', 'تعرف', 'بانها', 'كاينات', 'حية'...","['عرف', 'كتر', 'بان', 'كين', 'حية', 'دقق']","['كتر', 'عرف', 'بان', 'كين', 'حية', 'دقق']",اتعرف البكتيريا بانها كاينات حية دقيقة,عرف كتر بان كين حية دقق,البكتيريا تعرف بانها كاينات حية دقيقة,كتر عرف بان كين حية دقق
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.,ايتكون الهوا اساسا من النيتروجين,الهوا يتكون اساسا من النيتروجين,ايتكون الهوا اساسا من النيتروجين؟,الهوا يتكون اساسا من النيتروجين.,ايتكون الهوا اساسا من النيتروجين؟,الهوا يتكون اساسا من النيتروجين.,"['ايتكون', 'الهوا', 'اساسا', 'من', 'النيتروجين']","['الهوا', 'يتكون', 'اساسا', 'من', 'النيتروجين']","['ايتكون', 'الهوا', 'اساسا', 'من', 'النيتروجين']","['الهوا', 'يتكون', 'اساسا', 'من', 'النيتروجين']","['ايت', 'هوا', 'سسا', 'من', 'ترج']","['هوا', 'يتك', 'سسا', 'من', 'ترج']",ايتكون الهوا اساسا من النيتروجين,ايت هوا سسا من ترج,الهوا يتكون اساسا من النيتروجين,هوا يتك سسا من ترج


In [ ]:
label_encoder = LabelEncoder()
data["label"] = label_encoder.fit_transform(data["Category"])
num_labels = len(label_encoder.classes_)
print(num_labels)
print(label_encoder.classes_)

17
['الاقتصاد والعمل' 'البيئة والطاقة' 'البيولوجيا' 'التاريخ' 'الترفيه'
 'التطوع' 'التعليم' 'التكنولوجيا' 'الثقافة' 'الجغرافيا' 'الدين' 'الرياضة'
 'السفر والسياحة' 'السياسة والقانون' 'الصحة' 'العلوم' 'علم الاجتماع']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data["Question_for_classification"],data["label"],test_size=0.2,random_state=42,stratify=data["label"])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
train_encodings = tokenizer(X_train.tolist(),truncation=True,padding=True,max_length=128)
test_encodings = tokenizer(X_test.tolist(),truncation=True,padding=True,max_length=128)

In [ ]:
train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': y_train.tolist()
})

test_dataset = Dataset.from_dict({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask'],
    'labels': y_test.tolist()
})

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("aubmindlab/bert-base-arabertv02",num_labels=num_labels)

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

In [ ]:
training_args = TrainingArguments(
    output_dir='./bertresults',
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    optim="adamw_torch"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,1.360091
2,No log,0.947124
3,No log,0.856873


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=363, training_loss=1.2978033901246127, metrics={'train_runtime': 102.7468, 'train_samples_per_second': 112.587, 'train_steps_per_second': 3.533, 'total_flos': 166473048792960.0, 'train_loss': 1.2978033901246127, 'epoch': 3.0})

In [ ]:
predictions = trainer.predict(test_dataset)
y_pred = predictions.predictions.argmax(axis=1)

In [ ]:
print("BERT Results")
print("Accuracy",accuracy_score(y_test, y_pred))
print("Precision",precision_score(y_test, y_pred, average="weighted"))
print("Recall",recall_score(y_test, y_pred, average="weighted"))
print("F1-score",f1_score(y_test, y_pred, average="weighted"))

BERT Results
Accuracy 0.7502590673575129
Precision 0.7316657363617276
Recall 0.7502590673575129
F1-score 0.7349097123524632


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
X_qwen = np.load("/content/qwen_embeddings.npy")

In [ ]:
le = LabelEncoder()
y = le.fit_transform(data["Category"])

In [ ]:
X_train_qwen, X_test_qwen, y_train, y_test = train_test_split(X_qwen,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
best_model = SVC()
best_model.fit(X_train_qwen, y_train)

SVC()

In [ ]:
import joblib
joblib.dump(best_model, "qwen_svm_classifier.pkl")

['label_encoder.pkl']

In [ ]:
import joblib
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']